# XAI stability -- TESTS

In [ ]:
import sys

!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow
!{sys.executable} -m pip install torch

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
!{sys.executable} -m pip install -e OpenXAI

In [2]:
import time
import numpy as np
import pandas as pd
import nbimporter

# Utils
import torch
import os
import pickle
from sklearn.base import clone

import xgboost as xgb
from sklearn.neural_network import MLPClassifier


import XAI_stability_metrics as stab


import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
import openxai

# Data loaders
from openxai.dataloader import return_loaders

# Perturbation methods required for the computation of the relative stability metrics
from openxai.explainers.catalog.perturbation_methods import NormalPerturbation
from openxai.explainers.catalog.perturbation_methods import NewDiscrete_NormalPerturbation

# Perturbation definition

In [4]:
# Perturbation class parameters
perturbation_mean= 0.0
perturbation_std= 0.05
perturbation_flip_percentage= 0.01
    
perturbation= NormalPerturbation('tabular',
                                 mean=perturbation_mean,
                                 std_dev=perturbation_std,
                                 flip_percentage=perturbation_flip_percentage)

def generate_mask(explanation, top_k):
    mask_indices= torch.topk(explanation, top_k).indices
    mask= torch.zeros(explanation.shape) > 10
    for i in mask_indices:
        mask[i]= True
    return mask

# Loading data

# 1. Synthetic - 20 features - Numeric

In [9]:
ox_path= 'data/synth_OX_20/processed/'

train_ox= pd.read_csv(ox_path + 'X_train.csv')
test_ox = pd.read_csv(ox_path + 'X_test.csv')
labels_train_ox= pd.read_csv(ox_path + 'y_train.csv')
labels_test_ox = pd.read_csv(ox_path + 'y_test.csv')

In [14]:
# 2024-05-29 23:24:10,338 Best: 0.908399 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 1-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn1_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=256,
                            hidden_layer_sizes=(256,),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn1_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn1_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn1_model_ox.predict(test_ox))
acc_nn1_ox

0.83

In [15]:
# 2024-05-31 17:29:14,888 Best: 0.912651 using {'batch_size': 64, 'lr': 0.01, 'max_epochs': 64, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 2-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn2_model_ox= MLPClassifier(batch_size= 64,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn2_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn2_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn2_model_ox.predict(test_ox))
acc_nn2_ox

0.83

In [16]:
# 2024-06-03 16:27:56,409 Best: 0.905554 using {'batch_size': 16, 'lr': 0.01, 'max_epochs': 128, 
#                         'module__n_features': 20, 'module__n_neurons': 256, 'module__nonlin': ReLU()}

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ox= MLPClassifier(batch_size= 16,
                            learning_rate='constant', learning_rate_init= 0.01,
                            max_iter=128,
                            hidden_layer_sizes=(256, 256, 256),
                            activation='relu',
                            alpha=0.0001, random_state=0, solver='adam')

nn3_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn3_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn3_model_ox.predict(test_ox))
acc_nn3_ox

0.84

# 2. Adult Income

In [ ]:
ad_path= 'data/adult/processed/'

train_ad= pd.read_csv(ad_path + 'X_train.csv')
test_ad = pd.read_csv(ad_path + 'X_test.csv')
labels_train_ad= pd.read_csv(ad_path + 'y_train.csv')
labels_test_ad = pd.read_csv(ad_path + 'y_test.csv')

# 3. Chess (kr-vs-kp)

In [ ]:
ch_path= 'data/chess/processed/'

train_ch= pd.read_csv(ch_path + 'X_train.csv')
test_ch = pd.read_csv(ch_path + 'X_test.csv')
labels_train_ch= pd.read_csv(ch_path + 'y_train.csv')
labels_test_ch = pd.read_csv(ch_path + 'y_test.csv')

# 4. COMPAS

In [ ]:
cpas_path= 'data/compas/processed/'

train_cpas= pd.read_csv(cpas_path + 'X_train.csv')
test_cpas = pd.read_csv(cpas_path + 'X_test.csv')
labels_train_cpas= pd.read_csv(cpas_path + 'y_train.csv')
labels_test_cpas = pd.read_csv(cpas_path + 'y_test.csv')

# 5. Diabetes

In [ ]:
diab_path= 'data/diabetes/processed/'

train_diab= pd.read_csv(diab_path + 'X_train.csv')
test_diab = pd.read_csv(diab_path + 'X_test.csv')
labels_train_diab= pd.read_csv(diab_path + 'y_train.csv')
labels_test_diab = pd.read_csv(diab_path + 'y_test.csv')

# 6. German Credit

In [ ]:
ger_path= 'data/german/processed/'

train_ger= pd.read_csv(ger_path + 'X_train.csv')
test_ger = pd.read_csv(ger_path + 'X_test.csv')
labels_train_ger= pd.read_csv(ger_path + 'y_train.csv')
labels_test_ger = pd.read_csv(ger_path + 'y_test.csv')

# 7. HELOC

In [ ]:
hel_path= 'data/heloc/processed/'

train_hel= pd.read_csv(hel_path + 'X_train.csv')
test_hel = pd.read_csv(hel_path + 'X_test.csv')
labels_train_hel= pd.read_csv(hel_path + 'y_train.csv')
labels_test_hel = pd.read_csv(hel_path + 'y_test.csv')

# 8. HIGGS

In [ ]:
hig_path= 'data/higgs/processed/'

train_hig= pd.read_csv(hig_path + 'X_train.csv')
test_hig = pd.read_csv(hig_path + 'X_test.csv')
labels_train_hig= pd.read_csv(hig_path + 'y_train.csv')
labels_test_hig = pd.read_csv(hig_path + 'y_test.csv')

# 9. Independent

In [ ]:
indep_path= 'data/independent/processed/'

train_indep= pd.read_csv(indep_path + 'X_train.csv')
test_indep = pd.read_csv(indep_path + 'X_test.csv')
labels_train_indep= pd.read_csv(indep_path + 'y_train.csv')
labels_test_indep = pd.read_csv(indep_path + 'y_test.csv')

# 10. LSA - Law School Admission

In [ ]:
lsa_path= 'data/law_school_admission/processed/'

train_lsa= pd.read_csv(lsa_path + 'X_train.csv')
test_lsa = pd.read_csv(lsa_path + 'X_test.csv')
labels_train_lsa= pd.read_csv(lsa_path + 'y_train.csv')
labels_test_lsa = pd.read_csv(lsa_path + 'y_test.csv')